# Inspecting Datasets for Audit Errors Before Scraping

In [32]:
import yfinance as yf
import pandas as pd
import requests
from bs4 import BeautifulSoup

In [33]:
import warnings
# Ignore all warnings
warnings.filterwarnings("ignore", category=FutureWarning)

Sometimes, when trying to scrape the web for a dataset, you may encounter certain errors, like the `NoneType` Error. In webscraping using BeautifulSoup, if the code is expecting an HTML element but fails to find one, the find_all() object is `None`. 

It is, therefore, essential to ensure the code returns the expected elements from the web. For instance, working with the url: https://www.macrotrends.net/stocks/charts/TSLA/tesla/revenue, We realize that the code below returns the error: 

*AttributeError: 'NoneType' object has no attribute 'find_all'*


```
from bs4 import BeautifulSoup

url = 'https://www.macrotrends.net/stocks/charts/TSLA/tesla/revenue'
soup = BeautifulSoup(url, "html.parser")

table = soup.find("table", class_="prices")
rows = table.find_all("tr")
```
For the url used in this case, the error could most likely be occurring because `tbody` is not at the top level of teh parsed HTML that was downloaded.

This may be because the page contains several tables and nested structures. Therefore, searching for the first `tbody` globally, which the code essentially does, can fail or return teh wrong thing. 

It is, therefore, important to inspect what `BeautifulSoup` actually finds

In [34]:
url = 'https://www.macrotrends.net/stocks/charts/GME/gamestop/revenue'
headers = {'User-Agent':'Mozilla/5.0'}
response = requests.get(url, headers = headers)
print(response.status_code)

200


To actually see which tables `BeautfulSoup` actually sees, we'll run the following code.

It should allow us inspect available tables instead of guessing, then we can target the correct table appropriately.

In [35]:
beautiful_soup = BeautifulSoup(response.text,'html.parser')
tables = beautiful_soup.find_all("table")

print("Number of tables:", len(tables))

for i, table in enumerate(tables):
    print("\n----------------")
    print(f"Table {i}")
    print(table.get("class"))

Number of tables: 4

----------------
Table 0
['historical_data_table', 'table', 'table-sm', 'align-middle', 'text-center']

----------------
Table 1
['historical_data_table', 'table', 'table-sm', 'align-middle', 'text-center']

----------------
Table 2
['historical_data_table', 'table']

----------------
Table 3
['historical_data_table', 'table']


In [36]:
print(tables[0].prettify()[:500])

<table class="historical_data_table table table-sm align-middle text-center">
 <thead>
  <tr>
   <th colspan="2">
    GameStop Annual Revenue
    <br/>
    <span style="font-size:14px;">
     (Millions of US $)
    </span>
   </th>
  </tr>
 </thead>
 <tbody>
  <tr>
   <td>
    2026
   </td>
   <td>
    $3,630
   </td>
  </tr>
  <tr>
   <td>
    2025
   </td>
   <td>
    $3,823
   </td>
  </tr>
  <tr>
   <td>
    2024
   </td>
   <td>
    $5,273
   </td>
  </tr>
  <tr>
   <td>
    2023
   </td>
 


In [37]:
print(tables[1].prettify()[:500])

<table class="historical_data_table table table-sm align-middle text-center">
 <thead>
  <tr>
   <th colspan="2">
    GameStop Quarterly Revenue
    <br/>
    <span style="font-size:14px;">
     (Millions of US $)
    </span>
   </th>
  </tr>
 </thead>
 <tbody>
  <tr>
   <td>
    2026-04-30
   </td>
   <td>
    $835
   </td>
  </tr>
  <tr>
   <td>
    2026-01-31
   </td>
   <td>
    $1,104
   </td>
  </tr>
  <tr>
   <td>
    2025-10-31
   </td>
   <td>
    $821
   </td>
  </tr>
  <tr>
   <td>
  


In [38]:
print(tables[2].prettify()[:500])

<table class="historical_data_table table">
 <thead>
  <tr>
   <th style="text-align:center">
    Sector
   </th>
   <th style="text-align:center">
    Industry
   </th>
   <th style="text-align:center">
    Market Cap
   </th>
   <th style="text-align:center">
    Revenue
   </th>
  </tr>
 </thead>
 <tbody>
  <tr>
   <td style="text-align:center">
    <a href="https://www.macrotrends.net/stocks/sector/2/consumer-discretionary">
     Consumer Discretionary
    </a>
   </td>
   <td style="text-al


In [39]:
print(tables[3].prettify()[:500])

<table class="historical_data_table table">
 <thead>
  <tr>
   <th style="text-align:center; width:40%;">
    Stock Name
   </th>
   <th style="text-align:center; width:20%;">
    Country
   </th>
   <th style="text-align:center; width:20%;">
    Market Cap
   </th>
   <th style="text-align:center; width:20%;">
    PE Ratio
   </th>
  </tr>
 </thead>
 <tbody>
  <tr>
   <td style="text-align:left">
    <a href="/stocks/charts/NTDOY/nintendo/revenue">
     Nintendo (NTDOY)
    </a>
   </td>
   <td


From here, we now know that the Tesla datset with the records we need is in the second table in the second table denoted by [1] in teh above code block

It implies that using find('tbody') would only extract the first table with annual records. See below

In [40]:
print(beautiful_soup.find('tbody'))

<tbody>
<tr>
<td>2026</td>
<td>$3,630</td>
</tr>
<tr>
<td>2025</td>
<td>$3,823</td>
</tr>
<tr>
<td>2024</td>
<td>$5,273</td>
</tr>
<tr>
<td>2023</td>
<td>$5,927</td>
</tr>
<tr>
<td>2022</td>
<td>$6,011</td>
</tr>
<tr>
<td>2021</td>
<td>$5,090</td>
</tr>
<tr>
<td>2020</td>
<td>$6,466</td>
</tr>
<tr>
<td>2019</td>
<td>$8,285</td>
</tr>
<tr>
<td>2018</td>
<td>$8,547</td>
</tr>
<tr>
<td>2017</td>
<td>$7,965</td>
</tr>
<tr>
<td>2016</td>
<td>$9,364</td>
</tr>
<tr>
<td>2015</td>
<td>$9,296</td>
</tr>
<tr>
<td>2014</td>
<td>$9,040</td>
</tr>
<tr>
<td>2013</td>
<td>$8,887</td>
</tr>
<tr>
<td>2012</td>
<td>$9,551</td>
</tr>
</tbody>


The underlying lesson from the above inspection is that 

`find()` searches only what already exists in the parsed HTML.

If `find()` returns `None`, either:
    The selector is wrong, or 
    The page HTML downloaded by `requests` is different from what the browser rendered. 

In our case, Macrotrends likely gave us HTML different from what is visually visible in the browser 

We can now proceed by correctly identifying the table with Tesla quarterly records, which we intended to use for our analysis and dashboard creation.